# 01 Data Audit

This notebook validates the raw and processed Retailrocket data used by the project pipeline. It focuses on schema checks, event distribution, session behavior, and split readiness for next-item prediction.


## Goals

1. Confirm raw data quality and event composition.
2. Confirm processed session data is consistent with project assumptions.
3. Generate reusable summary artifacts and figures for reporting.


In [ ]:
import matplotlib
matplotlib.use("Agg")

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting defaults for readability.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

EVENTS_PATH = DATA_DIR / "events.csv"
SESSIONS_PATH = PROCESSED_DIR / "sessions.parquet"
TRAIN_PATH = PROCESSED_DIR / "train.parquet"
VALIDATION_PATH = PROCESSED_DIR / "validation.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"

print("Project root:", PROJECT_ROOT)
print("Events path exists:", EVENTS_PATH.exists())
print("Sessions path exists:", SESSIONS_PATH.exists())


In [ ]:
# Load raw events. Use all columns because the dataset schema is compact.
raw_events = pd.read_csv(EVENTS_PATH)

print("Raw events shape:", raw_events.shape)
print("Raw columns:", list(raw_events.columns))
raw_events.head()


In [ ]:
# Basic raw-level audit summary.
raw_summary = {
    "raw_rows": int(len(raw_events)),
    "raw_unique_visitors": int(raw_events["visitorid"].nunique()),
    "raw_unique_items": int(raw_events["itemid"].nunique()),
    "raw_event_counts": {str(k): int(v) for k, v in raw_events["event"].value_counts(dropna=False).items()},
    "raw_null_counts": {str(k): int(v) for k, v in raw_events.isnull().sum().items()},
}

raw_summary


In [ ]:
# Convert timestamp and inspect time coverage.
raw_events["timestamp_dt"] = pd.to_datetime(raw_events["timestamp"], unit="ms", utc=True)

print("Earliest event:", raw_events["timestamp_dt"].min())
print("Latest event:", raw_events["timestamp_dt"].max())
print("Span days:", (raw_events["timestamp_dt"].max() - raw_events["timestamp_dt"].min()).days)

# Event distribution chart.
event_counts = raw_events["event"].value_counts().sort_values(ascending=False)
ax = event_counts.plot(kind="bar", color=["#2a9d8f", "#e9c46a", "#e76f51"])
ax.set_title("Raw Event Type Counts")
ax.set_xlabel("Event Type")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "audit_event_type_counts.png", dpi=150)
plt.show()


In [ ]:
# Load processed sessions if available.
if not SESSIONS_PATH.exists():
    raise FileNotFoundError(
        "Missing processed sessions file. Run scripts/preprocess_data.py before this notebook."
    )

sessions = pd.read_parquet(SESSIONS_PATH)
print("Processed sessions shape:", sessions.shape)
print("Processed columns:", list(sessions.columns))
sessions.head()


In [ ]:
# Processed-session audit summary.
session_lengths = sessions.groupby("session_id").size().rename("session_length")

processed_summary = {
    "processed_rows": int(len(sessions)),
    "processed_sessions": int(sessions["session_id"].nunique()),
    "processed_unique_items": int(sessions["item_id"].nunique()),
    "processed_avg_session_length": float(session_lengths.mean()),
    "processed_median_session_length": float(session_lengths.median()),
    "processed_p95_session_length": float(session_lengths.quantile(0.95)),
}

processed_summary


In [ ]:
# Session-length distribution and item popularity diagnostics.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(session_lengths, bins=50, ax=axes[0], color="#457b9d")
axes[0].set_title("Session Length Distribution")
axes[0].set_xlabel("Session Length")
axes[0].set_ylabel("Session Count")

item_popularity = sessions["item_id"].value_counts()
axes[1].plot(np.arange(1, len(item_popularity) + 1), item_popularity.values)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Item Popularity (Log-Log)")
axes[1].set_xlabel("Item Rank")
axes[1].set_ylabel("Interactions")

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "audit_session_length_and_item_popularity.png", dpi=150)
plt.show()


In [ ]:
# Inspect split readiness if split files exist.
split_paths = {
    "train": TRAIN_PATH,
    "validation": VALIDATION_PATH,
    "test": TEST_PATH,
}

split_summary = {}
for split_name, path in split_paths.items():
    if path.exists():
        df = pd.read_parquet(path)
        split_summary[split_name] = {
            "rows": int(len(df)),
            "sessions": int(df["session_id"].nunique()),
            "unique_items": int(df["item_id"].nunique()),
            "time_min": str(df["session_start_ts"].min()),
            "time_max": str(df["session_start_ts"].max()),
        }

split_summary


In [ ]:
# Save a compact audit artifact for documentation.
audit_payload = {
    "raw_summary": raw_summary,
    "processed_summary": processed_summary,
    "split_summary": split_summary,
}

out_json = OUTPUT_TABLES / "data_audit_from_notebook.json"
with out_json.open("w", encoding="utf-8") as fp:
    json.dump(audit_payload, fp, indent=2, sort_keys=True)

print("Wrote:", out_json)
audit_payload


## Interpretation Checklist

Use this checklist before model training:

1. Session count and session length look plausible for next-item learning.
2. Event-type balance matches your selected filtering policy (`view` by default).
3. Split boundaries are chronological and non-overlapping.
4. Long-tail sparsity is acknowledged in evaluation interpretation.
